In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)
random.seed(42)

# Configuration
NUM_EMPLOYEES = 200
NUM_TASKS = 2000

# Generate employees
departments = ['Engineering', 'Sales', 'HR', 'Marketing', 'Finance', 'Support']
roles = {
    'Engineering': ['Junior Dev', 'Senior Dev', 'Tech Lead'],
    'Sales': ['Sales Rep', 'Account Manager', 'Sales Lead'],
    'HR': ['HR Coordinator', 'HR Manager', 'Recruiter'],
    'Marketing': ['Content Writer', 'SEO Specialist', 'Marketing Manager'],
    'Finance': ['Analyst', 'Accountant', 'Finance Manager'],
    'Support': ['Support Agent', 'Support Lead', 'Customer Success']
}

employees = []
for emp_id in range(1, NUM_EMPLOYEES + 1):
    dept = random.choice(departments)
    role = random.choice(roles[dept])
    join_date = fake.date_between(start_date='-5y', end_date='-6m')
    
    employees.append({
        'employee_id': f'EMP{emp_id:04d}',
        'name': fake.name(),
        'department': dept,
        'role': role,
        'join_date': join_date,
        'years_experience': round(random.uniform(0.5, 15), 1),
        'manager_id': f'EMP{random.randint(1, 20):04d}',
        'location': random.choice(['Bangalore', 'Mumbai', 'Delhi', 'Hyderabad', 'Remote'])
    })

employees_df = pd.DataFrame(employees)

# Generate tasks
task_categories = ['Bug Fix', 'Feature Dev', 'Code Review', 'Documentation', 
                   'Meeting', 'Testing', 'Client Call', 'Report', 'Training']
priorities = ['Low', 'Medium', 'High', 'Critical']

tasks = []
for task_id in range(1, NUM_TASKS + 1):
    emp = employees_df.sample(1).iloc[0]
    priority = random.choices(priorities, weights=[30, 40, 20, 10])[0]
    
    assigned_date = fake.date_between(start_date='-1y', end_date='today')
    deadline_days = random.randint(1, 14)
    deadline = assigned_date + timedelta(days=deadline_days)
    
    # Performance varies by department (adds realism)
    if emp['department'] == 'Engineering':
        completion_prob = 0.82
        efficiency_mean = 75
    elif emp['department'] == 'Sales':
        completion_prob = 0.91
        efficiency_mean = 85
    else:
        completion_prob = 0.88
        efficiency_mean = 80
    
    completed = random.random() < completion_prob
    
    if completed:
        # Some finish early, some late
        actual_days = deadline_days + random.randint(-2, 5)
        actual_days = max(1, actual_days)
        completion_date = assigned_date + timedelta(days=actual_days)
        deadline_met = completion_date <= deadline
    else:
        completion_date = None
        deadline_met = False
        actual_days = None
    
    hours_estimated = random.randint(2, 40)
    hours_actual = hours_estimated * random.uniform(0.5, 2.0) if completed else None
    efficiency_score = round(np.random.normal(efficiency_mean, 10), 1) if completed else None
    
    tasks.append({
        'task_id': f'TASK{task_id:05d}',
        'employee_id': emp['employee_id'],
        'department': emp['department'],
        'task_category': random.choice(task_categories),
        'priority': priority,
        'assigned_date': assigned_date,
        'deadline': deadline,
        'completion_date': completion_date,
        'completed': completed,
        'deadline_met': deadline_met,
        'hours_estimated': hours_estimated,
        'hours_actual': round(hours_actual, 1) if hours_actual else None,
        'efficiency_score': min(100, max(0, efficiency_score)) if efficiency_score else None,
        'satisfaction_rating': random.randint(1, 5) if completed else None
    })

tasks_df = pd.DataFrame(tasks)

# Save both datasets
employees_df.to_csv('../data/raw/employees.csv', index=False)
tasks_df.to_csv('../data/raw/tasks.csv', index=False)

print(f"Generated {len(employees_df)} employees and {len(tasks_df)} tasks")
print(employees_df.head())
print(tasks_df.head())

Generated 200 employees and 2000 tasks
  employee_id                name department              role   join_date  \
0     EMP0001         Robin Doyle    Support     Support Agent  2025-05-21   
1     EMP0002        Anthony Reed      Sales         Sales Rep  2022-02-09   
2     EMP0003  Christopher Barber    Finance        Accountant  2024-04-21   
3     EMP0004       Garrett Lynch      Sales        Sales Lead  2025-01-08   
4     EMP0005    Samantha Lindsey    Support  Customer Success  2025-06-06   

   years_experience manager_id   location  
0               0.9    EMP0009     Mumbai  
1              11.2    EMP0018  Bangalore  
2               1.0    EMP0003     Mumbai  
3               9.2    EMP0018     Mumbai  
4              10.7    EMP0014     Mumbai  
     task_id employee_id   department  task_category  priority assigned_date  \
0  TASK00001     EMP0096           HR       Training    Medium    2025-11-20   
1  TASK00002     EMP0200  Engineering    Client Call    Medium    20